# RAG

## 1. Load Documents

In [15]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_core.documents import Document
from pathlib import Path

def load_documents(dir: str) -> list[Document]:
    documents = []
    for file in list(Path(dir).glob("*")):
        if file.suffix.lower() == '.pdf':
            print(f"Loaded: {file}")
            documents += PyPDFLoader(file).load()
        elif file.suffix.lower() == ".docx":
            print(f"Loaded: {file}")
            documents += Docx2txtLoader(file).load()
        elif file.suffix.lower() in [".txt", ".rst", ".md"]:
            print(f"Loaded: {file}")
            documents += TextLoader(file).load()
        else:
            pass

    return documents


# load
directory = "./context_files"
documents = load_documents(directory)

Loaded: context_files/README.rst


In [16]:
print(documents)

[Document(metadata={'source': 'context_files/README.rst'}, page_content="################################\nEffective Intensity Computations\n################################\n\n\nIntroduction\n############\n\nEffective luminous intensity is a concept used in photometry to\nquantify the perceived brightness of a light source, particularly in\nthe context of flashing lights or other time-varying light sources, which\nare commonly used in signaling applications such as aviation, marine\nnavigation, and land transportation.\nIt represents the intensity (also called effective intensity, measured by cd) \nof a steady light source that would appear equally bright to the human eye as \nthe actual flashing or varying light source.\nTo calculate ELI, various mathematical models are applied to data collected \nunder controlled conditions. These models consider factors such as peak intensity, \nflash duration, inter-flash interval, and the human eye's response to flashing lights. \nDifferent stand

## 2. Split Documents

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
)

document_chunks = text_splitter.split_documents(documents)

## 3. Embedding

In [18]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# test the embedding function
document_embeddings = embedding_function.embed_documents(
    [chunk.page_content for chunk in document_chunks]
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4486.77it/s]


## 4. Store Embeddings

Optional, but recommanded.

In [19]:
from langchain_community.vectorstores import Chroma

vector_store = Chroma.from_documents(
    collection_name="my_rag_db",
    documents=document_chunks,
    embedding=embedding_function,
    persist_directory="./chroma_db"
)

# test similarity search
response = vector_store.similarity_search("What's Effective intensity?")
print(response)

[Document(metadata={'source': 'context_files/README.rst'}, page_content='Effective luminous intensity is a concept used in photometry to\nquantify the perceived brightness of a light source, particularly in\nthe context of flashing lights or other time-varying light sources, which\nare commonly used in signaling applications such as aviation, marine\nnavigation, and land transportation.\nIt represents the intensity (also called effective intensity, measured by cd) \nof a steady light source that would appear equally bright to the human eye as'), Document(metadata={'source': 'context_files/README.rst'}, page_content='Effective luminous intensity is a concept used in photometry to\nquantify the perceived brightness of a light source, particularly in\nthe context of flashing lights or other time-varying light sources, which\nare commonly used in signaling applications such as aviation, marine\nnavigation, and land transportation.\nIt represents the intensity (also called effective intensi

## 5. Retriever

In [20]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# test retriever
response = retriever.invoke("What's effective intensity?")
print(response)

[Document(metadata={'source': 'context_files/README.rst'}, page_content='Effective luminous intensity is a concept used in photometry to\nquantify the perceived brightness of a light source, particularly in\nthe context of flashing lights or other time-varying light sources, which\nare commonly used in signaling applications such as aviation, marine\nnavigation, and land transportation.\nIt represents the intensity (also called effective intensity, measured by cd) \nof a steady light source that would appear equally bright to the human eye as'), Document(metadata={'source': 'context_files/README.rst'}, page_content='Effective luminous intensity is a concept used in photometry to\nquantify the perceived brightness of a light source, particularly in\nthe context of flashing lights or other time-varying light sources, which\nare commonly used in signaling applications such as aviation, marine\nnavigation, and land transportation.\nIt represents the intensity (also called effective intensi

## 6. LLM

In [21]:
from langchain_groq import ChatGroq
from decouple import Config, RepositoryEnv
env_config = Config(RepositoryEnv("./.env"))

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    # model="llama-3.1-8b-instant",
    temperature=1.0,
    max_retries=2,
    api_key=env_config("GROQ_API_KEY"),
)

# test llm
query = "What's effective intensity?"
response = llm.invoke(query)
print(response)

content="Effective intensity is a term used to describe the perceived or relative intensity of a stimulus, rather than its absolute or physical intensity. In other words, it's the degree to which a stimulus is perceived or felt by an individual, taking into account factors such as their sensitivity, attention, and past experiences.\n\nFor example, two people may be exposed to the same level of noise, but one person may find it much more disturbing or intense than the other due to differences in their hearing sensitivity or personal tolerance. In this case, the effective intensity of the noise is higher for the person who finds it more disturbing, even though the physical intensity of the noise is the same for both people.\n\nEffective intensity can be influenced by a range of factors, including:\n\n1. Sensory sensitivity: People with greater sensitivity to a particular stimulus (e.g., light, sound, touch) may perceive it as more intense.\n2. Attention: Focusing attention on a stimulus 

## 7. Input - Output

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
    {context}
    Question: {question}
    Answer:"""
)

def documents2str(documents):
    return "\n\n".join(d.page_content for d in documents)


## 7. Chain

In [23]:
chain = (
    {"context": retriever | documents2str,
     "question": RunnablePassthrough()}
    | template
    | llm
    | StrOutputParser()
)

## Test

In [24]:
user_input = "What is effective intensity?"
response = chain.invoke(user_input)
print(response)

Effective intensity is the intensity of a steady light source, measured in candela (cd), that would appear equally bright to the human eye as a time-varying light source, such as a flashing light.
